In [ ]:
import polars as pl
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import math
from procompa import get_project_root, get_data_dir

PRJ_ROOT = get_project_root()
data_dir = PRJ_ROOT / "data"
data_dir_YM = get_data_dir() / "26.03_yeast.MAP"

Select High confidence Score

In [5]:
YM_COMPLEX_overlap = pl.read_csv(data_dir /"Complex_Portal/YeastMap/complex_db_complete_unique_overlap_complex_yeastmap_by_complex.csv")
complex_db_entries = pl.read_csv(data_dir / "Complex_Portal/Saccharomyces_cerevisiae_ComplexTab.tsv", separator= "\t" )

In [7]:
#firstly focus on predition taht have an extremly high confidence score of 1
#high_confidence_predictions = YM_COMPLEX_overlap.filter(pl.col("ComplexConfidence") == 1)

#add stoichiometry information from complex database (if given)
YM_COMPLEX_overlap = (
    YM_COMPLEX_overlap.join(
        complex_db_entries.select([
            "#Complex ac", 
            "Identifiers (and stoichiometry) of molecules in complex"
        ]),
        on="#Complex ac",
        how="left"
    )
    .with_columns(
        stoichiometry_known = ~pl.col("Identifiers (and stoichiometry) of molecules in complex").str.contains(r"\(0\)"),
        solely_proteins = ~pl.col("Identifiers (and stoichiometry) of molecules in complex").str.contains(r":")
    )
)

for which homdimers do i have the pdb file

In [12]:
homodimer_with_pdb = pl.read_csv(data_dir/ "Pipeline/merged_pdbs/pdb_file_names.csv")

In [14]:
homodimer_set = homodimer_with_pdb["uniprot_id_homodimer"].to_list()

YM_COMPLEX_overlap = YM_COMPLEX_overlap.with_columns(
    pl.col("true_complex").str.split(" ").alias("_proteins")
).with_columns(
    pl.col("_proteins")
      .list.eval(pl.element().is_in(homodimer_set))
      .list.sum()
      .alias("n_proteins_with_homodimer_pdb"),
    pl.col("_proteins")
      .list.eval(pl.element().is_in(homodimer_set))
      .list.all()
      .alias("all_proteins_have_pdb"),
).drop("_proteins")

strict pipeline dataset for complexes where all proteins

In [15]:
pdb_pipeline_dataset = YM_COMPLEX_overlap.filter((pl.col("solely_proteins") == True)& (pl.col("stoichiometry_known") == True)& pl.col("all_proteins_have_pdb")== True)

In [16]:
pdb_pipeline_dataset.write_csv(data_dir/ "Pipeline/pdb_present_first_setup_pipeline_complexes.csv")

initial first subset for the pipeline

In [4]:
pipeline_complexes = high_confidence_predictions.filter((pl.col("solely_proteins") == True)& (pl.col("stoichiometry_known") == True)& (pl.col("jaccard_similarity") == 1))

In [ ]:
pipeline_complexes.write_csv(data_dir/ "Pipeline/first_setup_pipeline_complexes.csv")

In [ ]:
uniprot_id_complexes_pipeline = pipeline_complexes.get_column("predicted_complex").str.split(" " ).explode().unique().to_list()

#pl.DataFrame({"uniprot_id": uniprot_id_complexes_pipeline}).write_csv(data_dir /"Pipeline/uniprot_ids_first_setup.csv")

other stuff

In [10]:
yeastmap_complex_pairs_with_scores = pl.read_csv("./Dataframes/Yeast_Map/yeastmap_complex_pairs_with_scores_incl_db.csv")
complex_by_breaking_values = pl.read_csv("./Dataframes/Yeast_Map/complex_by_breaking_values.csv")
YM_pred = pl.read_csv(data_dir_YM / "yeast.MAP_complexes_wConfidenceScores_wGenenames_total779_20251214.csv")
complex_db_entries = pl.read_csv(data_dir / "Complex_Portal/Saccharomyces cerevisiae_ComplexTab.tsv", separator= "\t" )


In [7]:
YM_pred_with_index = (
    YM_pred.with_row_index("predicted_complex_id", offset = 1)
    .with_columns(
        pl.format("CPX_{}", pl.col("predicted_complex_id")).alias("predicted_complex_id"))
    .rename({"UniProt_ACCs": "predicted_complex"})
)

shoudk have 168 complexes with high confidence scores, for how many of them do I have a complex on COMPLEX (so complete overlap) for how many of the, do i have a stoichioemtry, for how many of them do I have a structure? what is theor iptm score

In [4]:
pred_high_confidence_list = (YM_pred_with_index.filter(pl.col("ComplexConfidence") == 1).get_column("predicted_complex_id").unique().to_list())

In [5]:
high_confidence_complex_by_breaking_values = complex_by_breaking_values.filter(pl.col("predicted_complex_id").is_in(pred_high_confidence_list))

In [6]:
high_confidence_complex_by_breaking_values.height

168

## Do i have the nexessary AF data to run CF?

In [ ]:
pairs = pl.read_parquet(
    get_data_dir() / "25.12_pooled-ppi-yeast/data-26.03/summary_pairs.parquet"
) #takes around 1 min to load, as confirmed by jürgen the files from 26.04 also do dont contain the self pairs


In [3]:
self_pairs = pairs.filter(pl.col("af3_id1") == pl.col("af3_id2"))
print(f"Total self-pairs: {self_pairs.height}")


Total self-pairs: 0
